## Data Extraction

In this section we are going to extract 3 years of candle data from MT5 (Metra Trader 5). The available timeframe is M15 so we are going to resample to H1.

In [1]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime

# Connecting to MT5
if not mt5.initialize():
    print("Error while connecting to MT5")
    mt5.shutdown()

print(f"MT5 connected: {mt5.version()}")

# Extract historical data BTC/USD 1H
# 3 years of data
rates = mt5.copy_rates_from(
    "BTCUSD",
    mt5.TIMEFRAME_H1,
    datetime(2022, 1, 1),
    datetime(2024, 12, 31)
)

# Convert to DataFrame
df = pd.DataFrame(rates)
df['time'] = pd.to_datetime(df['time'], unit='s')
df.set_index('time', inplace=True)

print(f"\nShape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst rows:")
print(df.head())
print(f"\nPeriod:")
print(f"  From: {df.index[0]}")
print(f"  To: {df.index[-1]}")

mt5.shutdown()

Error while connecting to MT5
MT5 connected: (0, 0, '')


SystemError: <built-in function copy_rates_from> returned a result with an exception set

## Load and clean Data

In this section we are going to load, review and clean the data.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Loading the data from CSV file
df = pd.read_csv(
    '../data/BTCUSD_M15_202301020000_202605091130.csv',
    sep='\t',                    # MT5 uses tabulation as separator
    parse_dates=[['<DATE>', '<TIME>']],
    index_col=0
)

# Cleaning column names
df.columns = df.columns.str.replace('<', '').str.replace('>', '')
df.index.name = 'datetime'

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst rows:")
print(df.head())
print("\nPeriod:")
print(f"  From: {df.index[0]}")
print(f"  To: {df.index[-1]}")

C:\Users\mad52\AppData\Local\Temp\ipykernel_14620\2065239886.py:6: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df = pd.read_csv(


Shape: (116131, 7)

Columns: ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']

First rows:
                         OPEN      HIGH       LOW     CLOSE  TICKVOL  VOL  \
datetime                                                                    
2023-01-02 00:00:00  16599.62  16605.81  16594.76  16595.43      951    0   
2023-01-02 00:15:00  16595.47  16597.31  16584.76  16591.49      973    0   
2023-01-02 00:30:00  16592.26  16596.26  16585.39  16593.42     1009    0   
2023-01-02 00:45:00  16592.69  16597.51  16590.76  16595.81     1222    0   
2023-01-02 01:00:00  16595.46  16622.23  16595.33  16613.65      683    0   

                     SPREAD  
datetime                     
2023-01-02 00:00:00     197  
2023-01-02 00:15:00     197  
2023-01-02 00:30:00     197  
2023-01-02 00:45:00     197  
2023-01-02 01:00:00     197  

Period:
  From: 2023-01-02 00:00:00
  To: 2026-05-09 11:30:00


### Resampling to H1

In this section we are going to resample M15 candles to get 1H candle, each 1H candle nees to hold 4 M15 candles aprox.

In [9]:
# Resamplear de 15M a 1H
df_1h = df.resample('1h').agg({
    'OPEN': 'first',    # first candle of the period
    'HIGH': 'max',      # maximum of the period
    'LOW': 'min',       # minimum of the period
    'CLOSE': 'last',    # last candle of the period
    'TICKVOL': 'sum',   # total volume of the period
    'VOL': 'sum',
    'SPREAD': 'mean'    # average spread
}).dropna()

print("Shape 1H:", df_1h.shape)
print("\nFirst rows:")
print(df_1h.head())
print("\nVerification — 4 candles of 15M = 1 candle of 1H:")
print(f"15M candles: {df.shape[0]}")
print(f"1H candles:  {df_1h.shape[0]}")
print(f"Ratio:     {df.shape[0] / df_1h.shape[0]:.2f}")

Shape 1H: (29069, 7)

First rows:
                         OPEN      HIGH       LOW     CLOSE  TICKVOL  VOL  \
datetime                                                                    
2023-01-02 00:00:00  16599.62  16605.81  16584.76  16595.81     4155    0   
2023-01-02 01:00:00  16595.46  16622.23  16595.33  16610.87     2835    0   
2023-01-02 02:00:00  16611.01  16623.26  16579.01  16582.76     2004    0   
2023-01-02 03:00:00  16582.01  16583.76  16546.01  16560.96     3713    0   
2023-01-02 04:00:00  16560.46  16586.76  16551.26  16581.51     3006    0   

                     SPREAD  
datetime                     
2023-01-02 00:00:00   197.0  
2023-01-02 01:00:00   197.0  
2023-01-02 02:00:00   197.0  
2023-01-02 03:00:00   197.0  
2023-01-02 04:00:00   197.0  

Verification — 4 candles of 15M = 1 candle of 1H:
15M candles: 116131
1H candles:  29069
Ratio:     4.00


Perfect, we can see that each one hour candle holds 4 M15 candles.

### Saving processed data

In [7]:
import os

os.makedirs('../data', exist_ok=True)
df_1h.to_csv('../data/BTCUSD_1H.csv')

print("Data saved: ../data/BTCUSD_1H.csv")
print(f"Final shape: {df_1h.shape}")

Data saved: ../data/BTCUSD_1H.csv
Final shape: (29069, 7)


## Data extraction

Source: MetaTrader 5 — real historical data
Original timeframe: 15M (116,131 candles)
Final Timeframe: 1H (29,069 candles) — resampled using OHLCV
Period: 2023-01-02 → 2026-05-09 (~3.5 years)

Resample:
- OPEN  → first candle of the period
- HIGH  → period maximum
- LOW   → period minimum
- CLOSE → last candle of the period
- VOL   → period total volume